# Lakeview Dashboard Migration - Volume-Based Approach

This notebook implements a complete file-based migration workflow for Databricks Lakeview dashboards.

## Features
- Export dashboards as .lvdash.json files to Databricks volumes
- Transform catalog/schema/table/volume references using CSV mappings
- Capture and restore dashboard permissions (best effort)
- Support for OAuth, Service Principal, and PAT authentication
- Two workflow options: Manual import or Automated import
- Generate detailed migration reports

## Workflows

### Shared Steps (Both Workflows)
1. **Export** dashboards from source workspace (Cell 6)
2. **Transform** JSON using CSV mappings (Cell 7)

### Manual Import Workflow (Cells 8-10)
3. **Manual Import** - Import dashboards via Databricks UI
4. **Apply ACLs** - Run Cell 9 to apply permissions
5. **Generate Report** - Manual workflow report

### Automated Import Workflow (Cells 11-12)
3. **Automated Import** - Import dashboards programmatically
4. **Apply ACLs** - Apply permissions automatically
5. **Generate Report** - Automated workflow report

---

## Cell 1: Configuration & Imports

**Purpose:** Import required libraries and define configuration parameters.

**Instructions:**
1. Choose authentication method (OAuth RECOMMENDED)
2. Configure workspace URLs and credentials
3. Set volume paths (must exist in Unity Catalog)
4. Choose dashboard selection method and workflow type

In [ ]:
import json
import csv
import re
import os
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from pathlib import Path
import pandas as pd

try:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.workspace import ExportFormat, ImportFormat
    print("✅ Databricks SDK imported successfully")
except ImportError:
    print("⚠️  Warning: databricks-sdk not installed. Install with: %pip install databricks-sdk")

# ============================================================================
# AUTHENTICATION CONFIGURATION
# ============================================================================

# Choose authentication method: "pat", "oauth", "service_principal"
# RECOMMENDED: "oauth" for most use cases
AUTH_METHOD = "oauth"  # Default: OAuth (RECOMMENDED)

# --- OAuth Authentication (Azure AD) - RECOMMENDED ---
# Uses environment variables: ARM_CLIENT_ID, ARM_TENANT_ID, ARM_CLIENT_SECRET
# Or Azure CLI credentials (az login)
# RECOMMENDED for: Interactive use, temporary migrations, development, most scenarios
# Advantages: Easy setup, secure, no credential management, Azure-managed tokens
# Setup: Run 'az login' or set environment variables

# --- Service Principal Authentication ---
# Store credentials in Databricks secrets
# Good for: Production, automated pipelines, CI/CD, long-running jobs
# Setup: Create service principals in Azure AD, store in secrets
SOURCE_SP_CLIENT_ID = dbutils.secrets.get(scope="migration", key="source-sp-client-id")
SOURCE_SP_CLIENT_SECRET = dbutils.secrets.get(scope="migration", key="source-sp-secret")
SOURCE_SP_TENANT_ID = dbutils.secrets.get(scope="migration", key="source-sp-tenant")

TARGET_SP_CLIENT_ID = dbutils.secrets.get(scope="migration", key="target-sp-client-id")
TARGET_SP_CLIENT_SECRET = dbutils.secrets.get(scope="migration", key="target-sp-secret")
TARGET_SP_TENANT_ID = dbutils.secrets.get(scope="migration", key="target-sp-tenant")

# --- PAT Token Authentication ---
# Good for: Quick tests, development environments, legacy systems
# Note: Tokens expire and require rotation, less secure than OAuth
SOURCE_PAT_TOKEN = dbutils.secrets.get(scope="migration", key="source-token")
TARGET_PAT_TOKEN = dbutils.secrets.get(scope="migration", key="target-token")
# SOURCE_PAT_TOKEN = "dapi..."  # For testing only - DO NOT commit to version control
# TARGET_PAT_TOKEN = "dapi..."  # For testing only - DO NOT commit to version control

# ============================================================================
# WORKSPACE CONFIGURATION
# ============================================================================

# Source Workspace Configuration
SOURCE_WORKSPACE_URL = "https://dev-workspace.cloud.databricks.com"

# Target Workspace Configuration
TARGET_WORKSPACE_URL = "https://prod-workspace.cloud.databricks.com"

# Volume Paths (must exist and be accessible from both source and target)
VOLUME_BASE_PATH = "/Volumes/migration_cat/migration_schema/migration_vol/dashboard_migration"
MAPPING_CSV_PATH = f"{VOLUME_BASE_PATH}/mappings/catalog_schema_mapping.csv"
EXPORT_PATH = f"{VOLUME_BASE_PATH}/exported"
TRANSFORMED_PATH = f"{VOLUME_BASE_PATH}/transformed"
LOGS_PATH = f"{VOLUME_BASE_PATH}/logs"

# ============================================================================
# DASHBOARD SELECTION
# ============================================================================

# Choose one approach:
# Option 1: Explicit list of dashboard IDs
DASHBOARD_IDS = [
    # "abc123def456",
    # "ghi789jkl012",
]

# Option 2: Export all dashboards from a folder path
SOURCE_FOLDER_PATH = "/Workspace/Shared/Dashboards"
USE_FOLDER_PATH = True  # Set to False to use DASHBOARD_IDS instead

# Target Configuration
TARGET_FOLDER_PATH = "/Workspace/Shared/Migrated_Dashboards"

# ============================================================================
# MIGRATION OPTIONS
# ============================================================================

VALIDATE_QUERIES = False  # Set to True to validate queries before import
SKIP_PERMISSIONS = False  # Set to True to skip permissions migration
DRY_RUN = False          # Set to True to test without actually importing

# ============================================================================
# DISPLAY CONFIGURATION
# ============================================================================

print("✅ Configuration loaded")
print(f"\nAuthentication Method: {AUTH_METHOD}")
if AUTH_METHOD == "oauth":
    print("   Using OAuth/Azure AD (RECOMMENDED - Good for most use cases)")
elif AUTH_METHOD == "service_principal":
    print("   Using Service Principal (Good for production/automation)")
elif AUTH_METHOD == "pat":
    print("   Using PAT tokens (Good for quick tests)")

print(f"\nWorkspaces:")
print(f"   Source: {SOURCE_WORKSPACE_URL}")
print(f"   Target: {TARGET_WORKSPACE_URL}")
print(f"\nVolume: {VOLUME_BASE_PATH}")
print(f"Dry Run: {DRY_RUN}")

## Cell 2: Helper Functions - Authentication & Volume I/O

**Purpose:** Define authentication helper and volume I/O functions.

**Functions:**
- `create_workspace_client()` - Create WorkspaceClient with configured auth method
- `ensure_volume_directory()` - Create directories if they don't exist
- `read_volume_file()` - Read file content from volume
- `write_volume_file()` - Write content to volume file
- `list_volume_files()` - List files in volume directory
- `read_csv_from_volume()` - Read CSV file from volume

In [ ]:
def create_workspace_client(
    workspace_url: str, 
    is_source: bool = True
) -> WorkspaceClient:
    """
    Create WorkspaceClient with configured authentication method.
    
    Args:
        workspace_url: Databricks workspace URL
        is_source: True for source workspace, False for target
        
    Returns:
        Configured WorkspaceClient
    """
    if AUTH_METHOD == "service_principal":
        if is_source:
            return WorkspaceClient(
                host=workspace_url,
                client_id=SOURCE_SP_CLIENT_ID,
                client_secret=SOURCE_SP_CLIENT_SECRET,
                azure_tenant_id=SOURCE_SP_TENANT_ID
            )
        else:
            return WorkspaceClient(
                host=workspace_url,
                client_id=TARGET_SP_CLIENT_ID,
                client_secret=TARGET_SP_CLIENT_SECRET,
                azure_tenant_id=TARGET_SP_TENANT_ID
            )
    
    elif AUTH_METHOD == "oauth":
        # OAuth uses environment variables or Azure CLI
        return WorkspaceClient(host=workspace_url)
    
    elif AUTH_METHOD == "pat":
        token = SOURCE_PAT_TOKEN if is_source else TARGET_PAT_TOKEN
        return WorkspaceClient(host=workspace_url, token=token)
    
    else:
        raise ValueError(f"Unknown auth method: {AUTH_METHOD}")


def ensure_volume_directory(path: str) -> None:
    """
    Create volume directory if it doesn't exist.
    
    Args:
        path: Volume path to create
    """
    try:
        dbutils.fs.mkdirs(path)
        print(f"✅ Directory ensured: {path}")
    except Exception as e:
        print(f"⚠️  Directory creation warning: {e}")


def read_volume_file(volume_path: str) -> str:
    """
    Read file content from Databricks volume.
    
    Args:
        volume_path: Full path to file in volume
        
    Returns:
        File content as string
    """
    try:
        # Convert /Volumes path to local file system path
        local_path = volume_path.replace("/Volumes/", "/dbfs/Volumes/")
        with open(local_path, "r", encoding="utf-8") as f:
            content = f.read()
        return content
    except Exception as e:
        raise Exception(f"Failed to read volume file {volume_path}: {e}")


def write_volume_file(volume_path: str, content: str) -> None:
    """
    Write content to Databricks volume file.
    
    Args:
        volume_path: Full path to file in volume
        content: Content to write
    """
    try:
        # Ensure parent directory exists
        parent_dir = str(Path(volume_path).parent)
        ensure_volume_directory(parent_dir)
        
        # Write file
        local_path = volume_path.replace("/Volumes/", "/dbfs/Volumes/")
        with open(local_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"✅ Wrote file: {volume_path}")
    except Exception as e:
        raise Exception(f"Failed to write volume file {volume_path}: {e}")


def list_volume_files(volume_path: str, pattern: str = "*") -> List[str]:
    """
    List files in volume directory matching pattern.
    
    Args:
        volume_path: Volume directory path
        pattern: Glob pattern (e.g., "*.json")
        
    Returns:
        List of file paths
    """
    try:
        files = dbutils.fs.ls(volume_path)
        matched_files = []
        
        for file_info in files:
            file_path = file_info.path.replace("dbfs:", "")
            if pattern == "*" or file_path.endswith(pattern.replace("*", "")):
                matched_files.append(file_path)
                
        return matched_files
    except Exception as e:
        print(f"⚠️  Error listing files in {volume_path}: {e}")
        return []


def read_csv_from_volume(csv_path: str) -> List[Dict[str, str]]:
    """
    Read CSV file from volume and return as list of dictionaries.
    
    Args:
        csv_path: Path to CSV file in volume
        
    Returns:
        List of dictionaries with CSV data
    """
    try:
        content = read_volume_file(csv_path)
        lines = content.strip().split('\n')
        reader = csv.DictReader(lines)
        return list(reader)
    except Exception as e:
        raise Exception(f"Failed to read CSV from {csv_path}: {e}")


print("✅ Authentication and Volume I/O helper functions loaded")